# Archive - Backbone Experiment Backup Before Presentation Edit

Archived copy preserved for project history. Outputs are cleared and local/HPC paths are sanitized for the public repository.

**Portfolio note:** This notebook is part of a cleaned public portfolio version. Large datasets, trained model weights, HPC runtime artifacts, and private machine paths are not included.

**Public repository sections:**

- Archive Note


In [ ]:
# 1) Remove CPU wheels
%pip uninstall -y torch torchvision torchaudio

# 2) Install CUDA-enabled wheels (matches your driver/CUDA 13.x setup)
%pip install --index-url https://download.pytorch.org/whl/cu130 torch torchvision

In [ ]:
import torch, torchvision
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Cell 1: install
%pip install -U pip
%pip install --index-url https://download.pytorch.org/whl/cpu torch torchvision
%pip install timm albumentations scikit-learn pandas numpy matplotlib seaborn pillow


In [ ]:
# Cell 2: imports
import os
import time
import json
import math
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torchvision  # <- needed for torchvision.__version__
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)

import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries installed and imported successfully.")
print("PyTorch version:", torch.__version__)
print("torchvision version:", torchvision.__version__)
print("timm version:", timm.__version__)


In [ ]:
%pip install ipywidgets

## Section 1 — Imports and Device Setup

In [ ]:
import os
import time
import json
import math
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

import timm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)

import matplotlib.pyplot as plt
import seaborn as sns

print("PyTorch version:", torch.__version__)
print("timm version:", timm.__version__)

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = get_device()
print("Device:", DEVICE)

## Section 2 — Reproducibility

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def set_best_effort_determinism():
    try:
        if torch.cuda.is_available():
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(False)
    except Exception:
        pass

seed_everything(42)
set_best_effort_determinism()
print("Seeds initialized.")

## Section 3 — Config

In [ ]:
from dataclasses import dataclass
from pathlib import Path

# Make sure DEVICE is defined earlier, e.g.:
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

@dataclass
class CFG:
    # Paths
    data_root: str = "data/raw/Universal_Tomato_Dataset"
    train_dir: str = "train"
    val_dir: str = "val"
    test_dir: str = "test"

    # Model
    model_name: str = "tf_efficientnet_b4"
    num_classes: int = 16
    pretrained: bool = True

    # Image
    img_size: int = 300

    # Training
    epochs: int = 100
    batch_size: int = 8
    lr: float = 1e-4
    weight_decay: float = 1e-4
    label_smoothing: float = 0.05
    grad_clip: float = 1.0

    # Scheduler
    min_lr: float = 1e-6

    # Early stopping
    patience: int = 10

    # System
    num_workers: int = 0
    pin_memory: bool = True

    # General
    seed: int = 42
    device: str = DEVICE

    # Output
    out_root: str = "./trackB_outputs"

cfg = CFG()
Path(cfg.out_root).mkdir(parents=True, exist_ok=True)
print(cfg)


## Section 4 — Dataset Sanity Check

In [ ]:
def sanity_check_dataset(cfg: CFG):
    root = Path(cfg.data_root)
    train_path = root / cfg.train_dir
    val_path = root / cfg.val_dir
    test_path = root / cfg.test_dir

    print("Checking dataset paths...")
    print("Train path:", train_path, "| exists:", train_path.exists())
    print("Val path  :", val_path, "| exists:", val_path.exists())
    print("Test path :", test_path, "| exists:", test_path.exists())

    if not train_path.exists():
        raise FileNotFoundError(f"Missing train path: {train_path}")
    if not val_path.exists():
        raise FileNotFoundError(f"Missing val path: {val_path}")
    if not test_path.exists():
        raise FileNotFoundError(f"Missing test path: {test_path}")

    train_classes = sorted([p.name for p in train_path.iterdir() if p.is_dir()])
    val_classes = sorted([p.name for p in val_path.iterdir() if p.is_dir()])
    test_classes = sorted([p.name for p in test_path.iterdir() if p.is_dir()])

    print("Train classes:", len(train_classes))
    print("Val classes  :", len(val_classes))
    print("Test classes :", len(test_classes))

    if len(train_classes) != cfg.num_classes:
        raise ValueError(f"Expected {cfg.num_classes} classes in train, got {len(train_classes)}")

    if train_classes != val_classes or train_classes != test_classes:
        raise ValueError("Class folders differ across train / val / test.")

    print("\nClass names:")
    for c in train_classes:
        print("-", c)

    print("\nDataset sanity check passed.")

sanity_check_dataset(cfg)

## Section 5 — Augmentation Design

In [ ]:
def get_train_tfms(mode: str, img_size: int):
    """
    mode:
    - 'none'
    - 'safe'
    """
    if mode == "none":
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=(0.485, 0.456, 0.406),
                        std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

    if mode == "safe":
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.Affine(
                scale=(0.98, 1.02),
                translate_percent=(0.00, 0.03),
                rotate=(-8, 8),
                shear=(-3, 3),
                p=0.35
            ),
            A.RandomBrightnessContrast(
                brightness_limit=0.10,
                contrast_limit=0.10,
                p=0.25
            ),
            A.GaussianBlur(blur_limit=(3, 3), p=0.08),
            A.Normalize(mean=(0.485, 0.456, 0.406),
                        std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

    raise ValueError(f"Unknown augmentation mode: {mode}")


def get_eval_tfms(img_size: int):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

## Section 6 — Custom Dataset Wrapper

In [ ]:
class AlbumentationsImageFolder(ImageFolder):
    def __init__(self, root: str, albu_transform=None):
        super().__init__(root=root, transform=None)
        self.albu_transform = albu_transform

    def __getitem__(self, index):
        path, label = self.samples[index]

        from PIL import Image
        img = Image.open(path).convert("RGB")
        img = np.array(img)

        if self.albu_transform is not None:
            img = self.albu_transform(image=img)["image"]

        return img, label

## Section 7 — DataLoaders

In [ ]:
def build_datasets(cfg: CFG, aug_mode: str):
    root = Path(cfg.data_root)

    train_ds = AlbumentationsImageFolder(
        root / cfg.train_dir,
        albu_transform=get_train_tfms(aug_mode, cfg.img_size)
    )

    val_ds = AlbumentationsImageFolder(
        root / cfg.val_dir,
        albu_transform=get_eval_tfms(cfg.img_size)
    )

    test_ds = AlbumentationsImageFolder(
        root / cfg.test_dir,
        albu_transform=get_eval_tfms(cfg.img_size)
    )

    return train_ds, val_ds, test_ds


def build_loaders(cfg: CFG, aug_mode: str):
    train_ds, val_ds, test_ds = build_datasets(cfg, aug_mode)

    class_names = train_ds.classes
    if len(class_names) != cfg.num_classes:
        raise ValueError(f"Expected {cfg.num_classes} classes, got {len(class_names)}")

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        drop_last=False
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory
    )

    return train_loader, val_loader, test_loader, class_names


train_loader, val_loader, test_loader, class_names = build_loaders(cfg, "none")

print("Classes:", class_names)
print("Train steps:", len(train_loader))
print("Val steps  :", len(val_loader))
print("Test steps :", len(test_loader))

## Section 8 — Visual Audit of Augmentations

In [ ]:
def denormalize_image(tensor_img):
    mean = torch.tensor([0.485, 0.456, 0.406], device=tensor_img.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=tensor_img.device).view(3, 1, 1)
    img = tensor_img * std + mean
    img = torch.clamp(img, 0, 1)
    return img


def show_augmented_samples(cfg: CFG, aug_mode: str, n: int = 8):
    train_ds, _, _ = build_datasets(cfg, aug_mode)

    fig, axes = plt.subplots(2, n // 2, figsize=(16, 8))
    axes = axes.flatten()

    for ax in axes:
        idx = random.randint(0, len(train_ds) - 1)
        x, y = train_ds[idx]
        x = denormalize_image(x).permute(1, 2, 0).cpu().numpy()
        ax.imshow(x)
        ax.set_title(train_ds.classes[y], fontsize=9)
        ax.axis("off")

    plt.suptitle(f"Augmentation mode: {aug_mode}", fontsize=14)
    plt.tight_layout()
    plt.show()

show_augmented_samples(cfg, "safe", n=8)

## Section 9 — Model Builder

In [ ]:
def build_model(cfg: CFG):
    model = timm.create_model(
        cfg.model_name,
        pretrained=cfg.pretrained,
        num_classes=cfg.num_classes
    )
    return model

model = build_model(cfg).to(cfg.device)

xb, yb = next(iter(train_loader))
xb = xb.to(cfg.device)
with torch.no_grad():
    out = model(xb)

print("Input batch shape :", xb.shape)
print("Output logits shape:", out.shape)

## Section 10 — Metrics and Evaluation

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader, device, criterion):
    model.eval()

    all_targets = []
    all_preds = []
    all_probs = []

    loss_sum = 0.0
    total = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        logits = model(images)
        loss = criterion(logits, targets)

        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        loss_sum += loss.item() * targets.size(0)
        total += targets.size(0)

        all_targets.extend(targets.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.append(probs.cpu())

    avg_loss = loss_sum / max(total, 1)
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average="macro")

    return {
        "loss": avg_loss,
        "acc": acc,
        "macro_f1": macro_f1,
        "y_true": np.array(all_targets),
        "y_pred": np.array(all_preds),
        "probs": torch.cat(all_probs, dim=0).numpy() if len(all_probs) > 0 else None
    }

## Section 11 — Training Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, grad_clip=1.0):
    model.train()

    loss_sum = 0.0
    total = 0
    correct = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()

        if grad_clip is not None and grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()

        loss_sum += loss.item() * targets.size(0)
        total += targets.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == targets).sum().item()

    return {
        "loss": loss_sum / max(total, 1),
        "acc": correct / max(total, 1)
    }

## Section 12 — Report Saving Utilities

In [ ]:
def save_json(path: Path, data: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


def save_classification_outputs(out_dir: Path, y_true, y_pred, class_names):
    out_dir.mkdir(parents=True, exist_ok=True)

    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4,
        output_dict=True
    )

    report_text = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )

    cm = confusion_matrix(y_true, y_pred)

    with open(out_dir / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write(report_text)

    pd.DataFrame(report_dict).transpose().to_csv(out_dir / "classification_report.csv")
    np.save(out_dir / "confusion_matrix.npy", cm)

    return cm

## Section 13 — Experiment Runner

In [ ]:
def run_experiment(cfg: CFG, exp_name: str, aug_mode: str):
    print("\n" + "=" * 80)
    print(f"Starting experiment: {exp_name}")
    print(f"Augmentation mode: {aug_mode}")
    print("=" * 80)

    seed_everything(cfg.seed)
    set_best_effort_determinism()

    out_dir = Path(cfg.out_root) / exp_name
    out_dir.mkdir(parents=True, exist_ok=True)

    train_loader, val_loader, test_loader, class_names = build_loaders(cfg, aug_mode)

    model = build_model(cfg).to(cfg.device)

    criterion = nn.CrossEntropyLoss(
        label_smoothing=cfg.label_smoothing
    )

    optimizer = optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=cfg.epochs,
        eta_min=cfg.min_lr
    )

    history = []
    best_macro_f1 = -1.0
    best_epoch = -1
    patience_counter = 0

    best_model_path = out_dir / "best_model.pt"
    last_model_path = out_dir / "last_model.pt"

    for epoch in range(1, cfg.epochs + 1):
        start_time = time.time()

        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=cfg.device,
            grad_clip=cfg.grad_clip
        )

        val_metrics = evaluate_model(
            model=model,
            loader=val_loader,
            device=cfg.device,
            criterion=criterion
        )

        scheduler.step()
        epoch_time = time.time() - start_time
        current_lr = optimizer.param_groups[0]["lr"]

        row = {
            "epoch": epoch,
            "lr": current_lr,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["acc"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "val_macro_f1": val_metrics["macro_f1"],
            "time_sec": epoch_time
        }
        history.append(row)

        print(
            f"Epoch {epoch:03d}/{cfg.epochs} | "
            f"LR {current_lr:.2e} | "
            f"TrainLoss {train_metrics['loss']:.4f} | "
            f"TrainAcc {train_metrics['acc']:.4f} | "
            f"ValLoss {val_metrics['loss']:.4f} | "
            f"ValAcc {val_metrics['acc']:.4f} | "
            f"ValMacroF1 {val_metrics['macro_f1']:.4f} | "
            f"Time {epoch_time:.1f}s"
        )

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "cfg": asdict(cfg),
            "class_names": class_names
        }, last_model_path)

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            patience_counter = 0

            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "cfg": asdict(cfg),
                "class_names": class_names
            }, best_model_path)

            print(f"Best model updated at epoch {epoch} | Val Macro-F1 = {best_macro_f1:.4f}")
        else:
            patience_counter += 1
            print(f"No improvement | patience {patience_counter}/{cfg.patience}")

        if patience_counter >= cfg.patience:
            print(f"Early stopping triggered. Best epoch: {best_epoch}")
            break

    history_df = pd.DataFrame(history)
    history_df.to_csv(out_dir / f"{exp_name}_history.csv", index=False)

    ckpt = torch.load(best_model_path, map_location=cfg.device)
    model.load_state_dict(ckpt["model_state_dict"])

    val_final = evaluate_model(model, val_loader, cfg.device, criterion)
    test_final = evaluate_model(model, test_loader, cfg.device, criterion)

    val_cm = save_classification_outputs(
        out_dir / "val_reports",
        val_final["y_true"],
        val_final["y_pred"],
        class_names
    )

    test_cm = save_classification_outputs(
        out_dir / "test_reports",
        test_final["y_true"],
        test_final["y_pred"],
        class_names
    )

    summary = {
        "exp_name": exp_name,
        "aug_mode": aug_mode,
        "best_epoch": best_epoch,
        "best_val_macro_f1": float(best_macro_f1),
        "final_val_loss": float(val_final["loss"]),
        "final_val_acc": float(val_final["acc"]),
        "final_val_macro_f1": float(val_final["macro_f1"]),
        "final_test_loss": float(test_final["loss"]),
        "final_test_acc": float(test_final["acc"]),
        "final_test_macro_f1": float(test_final["macro_f1"]),
        "class_names": class_names
    }

    save_json(out_dir / "summary.json", summary)

    print("\nFinished experiment:", exp_name)
    print(json.dumps(summary, indent=2))

    return {
        "history_csv": str(out_dir / f"{exp_name}_history.csv"),
        "summary_json": str(out_dir / "summary.json"),
        "best_model": str(best_model_path),
        "history_df": history_df,
        "summary": summary,
        "val_cm": val_cm,
        "test_cm": test_cm
    }

## Section 14 — Track B Experiments

### Section 14A — Phase 1: Backbone Comparison

In [ ]:
cfg = CFG()
cfg.out_root = "./trackB_outputs"
Path(cfg.out_root).mkdir(parents=True, exist_ok=True)

print("Running Track B — Phase 1: Backbone Comparison")

results = []

for model_name in [
    "tf_efficientnet_b3",
    "tf_efficientnet_b4",
    "convnext_tiny"
]:
    cfg.model_name = model_name
    cfg.img_size = 224
    cfg.lr = 1e-4
    cfg.epochs = 100

    exp = run_experiment(
        cfg,
        exp_name=f"b1_{model_name}_224",
        aug_mode="safe"
    )
    results.append(exp)

### Section 14B — Phase 2: Resolution Comparison

In [ ]:
print("Running Track B — Phase 2: Resolution Comparison")

best_model = "tf_efficientnet_b3" # replace this after Phase 1

for img_size in [224, 300, 384]:
    cfg.model_name = best_model
    cfg.img_size = img_size
    cfg.lr = 1e-4
    cfg.epochs = 100

    exp = run_experiment(
        cfg,
        exp_name=f"b2_{best_model}_{img_size}",
        aug_mode="safe"
    )
    results.append(exp)

### Section 14C — Phase 3: LR Tuning

In [ ]:
print("Running Track B — Phase 3: LR Tuning (img_size=384)")

best_model = "tf_efficientnet_b3"

for lr in [3e-4, 1e-4, 5e-5]:
    cfg.model_name = best_model
    cfg.img_size = 384
    cfg.lr = lr
    cfg.epochs = 100

    exp = run_experiment(
        cfg,
        exp_name=f"b3_384_{best_model}_lr_{lr}",
        aug_mode="safe"
    )
    results.append(exp)

## Section 15 — Load All Track B Results

In [ ]:
trackb_summary_rows = [r["summary"] for r in results]
trackb_summary_df = pd.DataFrame(trackb_summary_rows)

trackb_summary_df = trackb_summary_df[[
    "exp_name",
    "aug_mode",
    "best_epoch",
    "final_val_acc",
    "final_val_macro_f1",
    "final_test_acc",
    "final_test_macro_f1"
]]

trackb_summary_df

In [ ]:
trackb_summary_df.to_csv(Path(cfg.out_root) / "trackB_summary.csv", index=False)
print("Saved:", Path(cfg.out_root) / "trackB_summary.csv")

## Section 16 — Plot Training Curves

In [ ]:
def plot_metric_comparison(dfs, labels, metric, title, ylabel):
    plt.figure(figsize=(10, 6))
    for df, label in zip(dfs, labels):
        plt.plot(df["epoch"], df[metric], label=label, linewidth=2)
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

In [ ]:
backbone_results = [r for r in results if r["summary"]["exp_name"].startswith("b1_")]
dfs_backbone = [pd.read_csv(r["history_csv"]) for r in backbone_results]
labels_backbone = [r["summary"]["exp_name"] for r in backbone_results]

plot_metric_comparison(
    dfs_backbone,
    labels_backbone,
    metric="val_macro_f1",
    title="Track B — Backbone Comparison (Val Macro-F1)",
    ylabel="Validation Macro-F1"
)

## Section 17 — Final Summary Table

In [ ]:
trackb_summary_df

## Section 18 — Final Bar Charts

In [ ]:
def barplot_summary(summary_df, metric, title):
    plt.figure(figsize=(10, 5))
    sns.barplot(data=summary_df, x="exp_name", y=metric)
    plt.title(title)
    plt.xlabel("Experiment")
    plt.ylabel(metric)
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
barplot_summary(trackb_summary_df, "final_val_acc", "Track B — Final Validation Accuracy")

In [ ]:
barplot_summary(trackb_summary_df, "final_val_macro_f1", "Track B — Final Validation Macro-F1")

In [ ]:
barplot_summary(trackb_summary_df, "final_test_acc", "Track B — Final Test Accuracy")

In [ ]:
barplot_summary(trackb_summary_df, "final_test_macro_f1", "Track B — Final Test Macro-F1")

## Section 19 — Confusion Matrix for Best Experiment

In [ ]:
best_idx = trackb_summary_df["final_test_macro_f1"].idxmax()
best_exp_name = trackb_summary_df.loc[best_idx, "exp_name"]
print("Best Track B experiment by test macro-F1:", best_exp_name)

In [ ]:
best_test_cm_path = Path(cfg.out_root) / best_exp_name / "test_reports" / "confusion_matrix.npy"
best_cm = np.load(best_test_cm_path)

plt.figure(figsize=(14, 12))
sns.heatmap(best_cm, cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title(f"Track B Confusion Matrix — {best_exp_name}")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()